# Exercises (Student) - MCP Client with LLM

In [ ]:
# Installation des dépendances
!pip install -q mcp nest_asyncio requests

In [ ]:
import os
from pathlib import Path

# Token pour le transport HTTP (si utilisé ultérieurement)
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")

# Passez à True si vous avez défini GITHUB_TOKEN pour utiliser un vrai LLM
USE_REAL_LLM = False

In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Permet d'utiliser asyncio dans des notebooks (Colab/Jupyter)
nest_asyncio.apply()

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

# Exercice : enregistrement de l'outil add
@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

# Exercice optionnel : outil multiply
@mcp.tool()
def multiply(a: int, b: int) -> int:
    "Multiply two numbers."
    return a * b

# Exercice : enregistrement de l'outil greet
@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hello {name}!"

if __name__ == "__main__":
    mcp.run()

## Exercise 1 (provide answer)

#Why is STDIO transport simple for local MCP dev compared to HTTP?
STDIO transport is simpler for local MCP development because it involves running the server as a subprocess and communicating directly via its standard input/output streams. This avoids the overhead and configuration complexities of setting up network sockets, handling HTTP requests/responses, and dealing with potential firewall issues that come with HTTP. It's a direct, in-process communication model that is ideal for testing and development on a single machine.

## Exercise 2

In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    # Exercice : créer StdioServerParameters pour lancer server.py
    params = StdioServerParameters(command="python", args=["server.py"])  # Command to run server.py
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            return session

In [ ]:
# In a new cell
session = await ex2_connect()
print("Exercise 2: OK (connected and initialized)")

## Exercise 3

In [ ]:
async def ex3_list():
    # Exercice : créer StdioServerParameters pour lancer server.py
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            # Exercice : lister les ressources
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            # Exercice : lister les outils
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))

In [ ]:
await ex3_list()

## Exercise 4

#Explain how the conversion to llm tool happens in MCP server code ?
The `convert_to_llm_tool` function takes an MCP `tool` object as input and transforms it into a dictionary format compatible with large language models (LLMs) that support function calling. Specifically:
- It sets the `type` to "function".
- It extracts the `name` and `description` from the MCP tool.
- It converts the `inputSchema` of the MCP tool into the `parameters` schema required by the LLM, including `properties` (argument names and their types) and `required` arguments. This makes the MCP tool discoverable and callable by the LLM.

In [ ]:
def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }

## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like "Add 2 to 20."

In [ ]:
import asyncio
import json
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

# Exercice 5 : simulateur LLM local (stub)
def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Planificateur stub qui choisit l'outil à partir du prompt."""
    prompt_lower = prompt.lower()
    calls: List[Dict[str, Any]] = []

    # Détection simple de l'opération et des nombres
    if any(word in prompt_lower for word in ["add", "plus", "sum"]):
        nums = re.findall(r"-?\d+", prompt)
        if len(nums) >= 2:
            print(f"finds {nums[0]} and {nums[1]} in prompt")
            calls.append({
                "name": "add",
                "args": {"a": int(nums[0]), "b": int(nums[1])},
            })
    elif any(word in prompt_lower for word in ["multiply", "times", "multiplie"]):
        nums = re.findall(r"-?\d+", prompt)
        if len(nums) >= 2:
            calls.append({
                "name": "multiply",
                "args": {"a": int(nums[0]), "b": int(nums[1])},
            })
    elif any(word in prompt_lower for word in ["hello", "greet", "bonjour"]):
        name = re.sub(r"(?i)(greet|hello|bonjour)\b.*", "", prompt).strip()
        name = name or "world"
        calls.append({
            "name": "greet",
            "args": {"name": name},
        })

    return calls


def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    """Appelle un vrai LLM si use_real=True, sinon utilise le stub local."""
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [ ]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    # Exercice : préparer les paramètres du serveur STDIO
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            # Exercice : lister les outils et les convertir au format LLM
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in tools.tools]
            # Exercice : obtenir les tool_calls depuis le LLM (ou le stub)
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])

In [ ]:
await ex5_run("Add 2 to 20")

In [ ]:
# Exercice optionnel : essayer multiply
await ex5_run("Multiply 3 by 4")

## Optional - add multiply(a, b) and rerun